In [119]:
from pyspark.sql import SparkSession
import datetime
import re
import os
import time
import pyspark
from pyspark.sql.types import StructType, StructField, BooleanType, DoubleType
from pyspark.sql.types import StringType
from pyspark.sql.types import DateType
from pyspark.sql.types import IntegerType
from pyspark.sql import Row
from datetime import date
import findspark
findspark.init()

os.environ['PYSPARK_PYTHON'] = 'python'
os.environ['PYSPARK_DRIVER_PYTHON'] ='jupyter'

In [26]:
spark = (
    SparkSession.builder
    .appName("SCD Type Demo")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.python.worker.reuse", "false")

    # ===== ALL PACKAGES IN ONE PLACE =====
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.spark:spark-token-provider-kafka-0-10_2.12:3.5.6",
            "org.postgresql:postgresql:42.7.7"
        ])
    )

    # ===== EXTENSIONS =====
    # .config(
    #     "spark.sql.extensions",
    #     "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
    #     "org.projectnessie.spark.extensions.NessieSparkSessionExtensions"
    # )


    .getOrCreate()
)

spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

print("Spark Session Started")

Spark Session Started


In [27]:


jdbc_url = "jdbc:postgresql://localhost:5432/scd_db"
connection_props = {
    "user": "user",
    "password": "password",
    "driver": "org.postgresql.Driver"
}


In [28]:
sc = spark.sparkContext
jvm = sc._jvm


DriverManager = jvm.java.sql.DriverManager
conn = DriverManager.getConnection(jdbc_url, connection_props["user"], connection_props["password"])
stmt = conn.createStatement()

create_sql = """
CREATE TABLE IF NOT EXISTS scd_example_one (
    id SERIAL PRIMARY KEY,
    name TEXT NOT NULL,
    dob DATE NOT NULL
);
"""

stmt.executeUpdate(create_sql)
stmt.close()
conn.close()


## SCD Type: 0

#### The Type 0 dimension attributes never change and are assigned to attributes that have durable values or are described as 'Original'. Examples: Date of Birth, Original Credit Score. Type 0 applies to most date dimension attributes [Wiki](https://en.wikipedia.org/wiki/Slowly_changing_dimension)

In [30]:

schema_0 = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("dob", DateType(), True),

])

data = [

    (1,'Jordan', date(1978,1,1)),
    (2, 'Joe', date(1979,1,1)),
    (3, 'Mack', date(1979,5,1)),
]

df = spark.createDataFrame(data, schema=schema_0)


df.show()

+---+------+----------+
| id|  name|       dob|
+---+------+----------+
|  1|Jordan|1978-01-01|
|  2|   Joe|1979-01-01|
|  3|  Mack|1979-05-01|
+---+------+----------+



In [31]:
df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_one") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .mode("append") \
    .save()

#### With this nothing is updated or upserted to

## SCD Type: 1

#### This method overwrites old with new data, and therefore does not track historical data.

In [34]:
sc = spark.sparkContext
jvm = sc._jvm


DriverManager = jvm.java.sql.DriverManager
conn = DriverManager.getConnection(jdbc_url, connection_props["user"], connection_props["password"])
stmt = conn.createStatement()

create_sql = """
CREATE TABLE IF NOT EXISTS scd_example_scd_typ_1 (

    player_id VARCHAR(10) NOT NULL PRIMARY KEY,
    AB int NOT NULL,
    H INT NOT NULL,
    BB INT NOT NULL,
    SO INT NOT NULL,
    RBI INT NOT NULL
);
"""

stmt.executeUpdate(create_sql)
stmt.close()
conn.close()


In [44]:
schema_1 = StructType([
    StructField("player_id", StringType(), nullable=False),
    StructField("ab", IntegerType(), nullable=False),
    StructField("h", IntegerType(), nullable=False),
    StructField("bb", IntegerType(), nullable=False),
    StructField("so", IntegerType(), nullable=False),
    StructField("rbi", IntegerType(), nullable=False),





])

data = [('60912', 400, 78, 12, 13, 34 ),
        ('60913', 500, 112, 45, 33, 67 ),
        ('60914', 101, 32, 7, 11, 10 ),
        ('60915', 234, 65, 64, 1, 38 )

]


df = spark.createDataFrame(data, schema=schema_1)



df.show()

+---------+---+---+---+---+---+
|player_id| ab|  h| bb| so|rbi|
+---------+---+---+---+---+---+
|    60912|400| 78| 12| 13| 34|
|    60913|500|112| 45| 33| 67|
|    60914|101| 32|  7| 11| 10|
|    60915|234| 65| 64|  1| 38|
+---------+---+---+---+---+---+



In [38]:
df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_scd_typ_1") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .mode("append") \
    .save()

In [45]:
df = (
    spark.read
  .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_scd_typ_1") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .load()
)

df.show(10)


+---------+---+---+---+---+---+
|player_id| ab|  h| bb| so|rbi|
+---------+---+---+---+---+---+
|    60912|400| 78| 12| 13| 34|
|    60913|500|112| 45| 33| 67|
|    60915|234| 65| 64|  1| 38|
|    60914|101| 32|  7| 11| 10|
+---------+---+---+---+---+---+



### In SCD Type I, on new updates to the data, the data is updated and the history isn't preserved

#### Unlike, iceberg or delta tables, for postgres this requires a staging table in postgres

In [46]:
new_data = [('60912', 450, 100, 18, 25, 50 ),
        ('60913', 501, 113, 45, 33, 68 ),

]


df = spark.createDataFrame(new_data, schema=schema_1)



df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_scd_typ_1_staging") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .mode("overwrite") \
    .save()


In [47]:
sc = spark.sparkContext
jvm = sc._jvm


DriverManager = jvm.java.sql.DriverManager
conn = DriverManager.getConnection(jdbc_url, connection_props["user"], connection_props["password"])
stmt = conn.createStatement()


upsert_sql = """
INSERT INTO scd_example_scd_typ_1 (player_id, AB, H, BB, SO, RBI)
SELECT player_id, AB, H, BB, SO, RBI
FROM scd_example_scd_typ_1_staging
ON CONFLICT (player_id)
DO UPDATE SET
  AB  = EXCLUDED.AB,
  H  = EXCLUDED.H,
  BB = EXCLUDED.BB,
  SO = EXCLUDED.SO,
  RBI = EXCLUDED.RBI
;
"""

stmt.executeUpdate(upsert_sql)
stmt.close()
conn.close()


In [48]:
df = (
    spark.read
  .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_scd_typ_1") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .load()
)

df.show(10)


+---------+---+---+---+---+---+
|player_id| ab|  h| bb| so|rbi|
+---------+---+---+---+---+---+
|    60915|234| 65| 64|  1| 38|
|    60914|101| 32|  7| 11| 10|
|    60912|450|100| 18| 25| 50|
|    60913|501|113| 45| 33| 68|
+---------+---+---+---+---+---+



### From here you can see that the values are able to be updated but the historical past values are completely lost

## SCD Type: 2

#### This method tracks historical data by creating multiple records for a given natural key in the dimensional tables with separate surrogate keys and/or different version numbers. Unlimited history is preserved for each insert.

In [109]:
sc = spark.sparkContext
jvm = sc._jvm


DriverManager = jvm.java.sql.DriverManager
conn = DriverManager.getConnection(jdbc_url, connection_props["user"], connection_props["password"])
stmt = conn.createStatement()

create_sql = """

DROP TABLE IF EXISTS scd_example_scd_typ_2;

CREATE TABLE IF NOT EXISTS scd_example_scd_typ_2 (

    player_id VARCHAR(10) NOT NULL,
    AB int NOT NULL,
    H INT NOT NULL,
    BB INT NOT NULL,
    SO INT NOT NULL,
    RBI INT NOT NULL,
    ACTIVE BOOLEAN NOT NULL DEFAULT TRUE,
    DATE_REC_CREATED DATE NOT NULL DEFAULT NOW(),
    DATE_REC_ENDED DATE,
    PRIMARY KEY(player_id, ACTIVE)
);
"""

stmt.executeUpdate(create_sql)
stmt.close()
conn.close()


In [110]:
schema_2 = StructType([
    StructField("player_id", StringType(), nullable=False),
    StructField("ab", IntegerType(), nullable=False),
    StructField("h", IntegerType(), nullable=False),
    StructField("bb", IntegerType(), nullable=False),
    StructField("so", IntegerType(), nullable=False),
    StructField("rbi", IntegerType(), nullable=False),
#StructField("active", BooleanType(), nullable=True),
  #  StructField("date_rec_created", DateType(), nullable=True),

])

data = [

    ('60912', 400, 78, 12, 13, 34 ),
    ('60913', 500, 112, 45, 33, 67 ),
    ('60914', 101, 32, 7, 11, 10  ),
    ('60915', 234, 65, 64, 1, 38  )

]




df = spark.createDataFrame(data, schema=schema_2)


df.show()

+---------+---+---+---+---+---+
|player_id| ab|  h| bb| so|rbi|
+---------+---+---+---+---+---+
|    60912|400| 78| 12| 13| 34|
|    60913|500|112| 45| 33| 67|
|    60914|101| 32|  7| 11| 10|
|    60915|234| 65| 64|  1| 38|
+---------+---+---+---+---+---+



In [111]:
df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_scd_typ_2") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .mode("append") \
    .save()

In [112]:
df = (
    spark.read
  .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_scd_typ_2") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .load()
)

df.show(10)


+---------+---+---+---+---+---+------+----------------+--------------+
|player_id| ab|  h| bb| so|rbi|active|date_rec_created|date_rec_ended|
+---------+---+---+---+---+---+------+----------------+--------------+
|    60913|500|112| 45| 33| 67|  true|      2026-01-22|          NULL|
|    60912|400| 78| 12| 13| 34|  true|      2026-01-22|          NULL|
|    60915|234| 65| 64|  1| 38|  true|      2026-01-22|          NULL|
|    60914|101| 32|  7| 11| 10|  true|      2026-01-22|          NULL|
+---------+---+---+---+---+---+------+----------------+--------------+



#### Notice here there is a true and date rec created fields to let you know what is the most recent version of their record, and the data this record was inserted

### Since this is postgres staging table again

In [113]:
new_data = [('60912', 450, 100, 18, 25, 50 ),
        ('60913', 501, 113, 45, 33, 68 ),
        ('70913', 123, 61, 1, 4,   42 ), # NEW RECORD FOR PLAYER ID


]


df = spark.createDataFrame(new_data, schema=schema_2)



df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_scd_typ_2_staging") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .mode("overwrite") \
    .save()


### MERGE Operation, also with setting the less recent record to not active

In [115]:
sc = spark.sparkContext
jvm = sc._jvm


DriverManager = jvm.java.sql.DriverManager
conn = DriverManager.getConnection(jdbc_url, connection_props["user"], connection_props["password"])
stmt = conn.createStatement()


upsert_sql = """
MERGE INTO scd_example_scd_typ_2 AS target
USING (
    SELECT
        player_id,
        AB, H, BB, SO, RBI
    FROM scd_example_scd_typ_2_staging
) AS source
ON target.player_id = source.player_id
   AND target.active = TRUE
WHEN MATCHED AND (
       target.AB  <> source.AB
    OR target.H   <> source.H
    OR target.BB  <> source.BB
    OR target.SO  <> source.SO
    OR target.RBI <> source.RBI
) THEN
    -- expire old row
    UPDATE SET
        active = FALSE,
        DATE_REC_ENDED = NOW()

WHEN NOT MATCHED THEN
    -- insert new version
    INSERT (
        player_id,
        AB, H, BB, SO, RBI
    )
    VALUES (
        source.player_id,
        source.AB, source.H, source.BB, source.SO, source.RBI
    );


INSERT INTO scd_example_scd_typ_2 (
    player_id,
    AB, H, BB, SO, RBI
)
SELECT
    s.player_id,
    s.AB, s.H, s.BB, s.SO, s.RBI
FROM scd_example_scd_typ_2_staging s
LEFT JOIN scd_example_scd_typ_2 t
  ON s.player_id = t.player_id
 AND t.active = TRUE
WHERE t.player_id IS NULL;

;
"""

stmt.executeUpdate(upsert_sql)
stmt.close()
conn.close()


In [116]:
df = (
    spark.read
  .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_scd_typ_2") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .load()
)

df.show(10)


+---------+---+---+---+---+---+------+----------------+--------------+
|player_id| ab|  h| bb| so|rbi|active|date_rec_created|date_rec_ended|
+---------+---+---+---+---+---+------+----------------+--------------+
|    60915|234| 65| 64|  1| 38|  true|      2026-01-22|          NULL|
|    60914|101| 32|  7| 11| 10|  true|      2026-01-22|          NULL|
|    60912|400| 78| 12| 13| 34| false|      2026-01-22|    2026-01-22|
|    70913|123| 61|  1|  4| 42|  true|      2026-01-22|          NULL|
|    60913|500|112| 45| 33| 67| false|      2026-01-22|    2026-01-22|
|    60912|450|100| 18| 25| 50|  true|      2026-01-22|          NULL|
|    60913|501|113| 45| 33| 68|  true|      2026-01-22|          NULL|
+---------+---+---+---+---+---+------+----------------+--------------+



#### As you can see from here, the newer records are indicated from a true value in the active column and a null date_rec_ended, this allows for you to see the historical values of a record while continually updating

# SCD: Type 3

#### This method tracks changes using separate columns and preserves limited history. The Type 3 preserves limited history as it is limited to the number of columns designated for storing historical data. The original table structure in Type 1 and Type 2 is the same but Type 3 adds additional columns. In the following example, an additional column has been added to the table to record the supplier's original state - only the previous history is stored.

In [149]:
sc = spark.sparkContext
jvm = sc._jvm


DriverManager = jvm.java.sql.DriverManager
conn = DriverManager.getConnection(jdbc_url, connection_props["user"], connection_props["password"])
stmt = conn.createStatement()

create_sql = """

DROP TABLE IF EXISTS scd_example_scd_typ_3;

CREATE TABLE IF NOT EXISTS scd_example_scd_typ_3 (

    player_id VARCHAR(10) NOT NULL,
    BA NUMERIC(4,3) NOT NULL,
    MOST_RECENT_BA NUMERIC(4,3) ,
    DATE_REC_EFF DATE NOT NULL DEFAULT NOW(),
    PRIMARY KEY(player_id)
);
"""

stmt.executeUpdate(create_sql)
stmt.close()
conn.close()


In [148]:


schema_3 = StructType([

    StructField("player_id", StringType(), False),
    StructField("ba", DoubleType(), False)
])



data = [

    ('67182', .234),
    ('67183', .308),
    ('67184', .412),
    ('67185', .112),
]


df = spark.createDataFrame(data, schema=schema_3)

df.show()

+---------+-----+
|player_id|   ba|
+---------+-----+
|    67182|0.234|
|    67183|0.308|
|    67184|0.412|
|    67185|0.112|
+---------+-----+



In [150]:
df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_scd_typ_3") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .mode("append") \
    .save()

In [151]:
df = (
    spark.read
  .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_scd_typ_3") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .load()
)

df.show(10)


+---------+-----+--------------+------------+
|player_id|   ba|most_recent_ba|date_rec_eff|
+---------+-----+--------------+------------+
|    67182|0.234|          NULL|  2026-01-22|
|    67184|0.412|          NULL|  2026-01-22|
|    67185|0.112|          NULL|  2026-01-22|
|    67183|0.308|          NULL|  2026-01-22|
+---------+-----+--------------+------------+



## Staging Table and this will be similar to an upsert

In [152]:
new_data = [(
    '67182', .278
),
    ('67183', .423),

    ('98192', .342)
]


df = spark.createDataFrame(new_data, schema=schema_3)

df.show()

+---------+-----+
|player_id|   ba|
+---------+-----+
|    67182|0.278|
|    67183|0.423|
|    98192|0.342|
+---------+-----+



In [153]:
df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_scd_typ_3_staging") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .mode("overwrite") \
    .save()

In [154]:
sc = spark.sparkContext
jvm = sc._jvm


DriverManager = jvm.java.sql.DriverManager
conn = DriverManager.getConnection(jdbc_url, connection_props["user"], connection_props["password"])
stmt = conn.createStatement()


upsert_sql = """
MERGE INTO scd_example_scd_typ_3 AS target
USING scd_example_scd_typ_3_staging AS source
ON target.player_id = source.player_id

WHEN MATCHED THEN
UPDATE SET
    MOST_RECENT_BA = target.BA,
    DATE_REC_EFF   = NOW(),
    BA             = source.BA


WHEN NOT MATCHED THEN
INSERT (
    player_id,
    BA,
    MOST_RECENT_BA,
    DATE_REC_EFF
)
VALUES (
    source.player_id,
    source.BA,
    NULL,
    NOW()
);


"""

stmt.executeUpdate(upsert_sql)
stmt.close()
conn.close()


In [155]:
df = (
    spark.read
  .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_scd_typ_3") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .load()
)

df.show(10)


+---------+-----+--------------+------------+
|player_id|   ba|most_recent_ba|date_rec_eff|
+---------+-----+--------------+------------+
|    67184|0.412|          NULL|  2026-01-22|
|    67185|0.112|          NULL|  2026-01-22|
|    67183|0.423|         0.308|  2026-01-22|
|    98192|0.342|          NULL|  2026-01-22|
|    67182|0.278|         0.234|  2026-01-22|
+---------+-----+--------------+------------+



#### Now this preserves partial history through the most recent column and it lets you know the last date the data has been updated

In [25]:
spark.stop()